# 📝 ReAct·멀티툴 에이전트 과제 LV3(통합)

> 이 단원의 개념을 하나로 묶는 **통합 과제 2문제**입니다. 도메인은 **구청 민원 안내**입니다.

## 풀이 방법
1. 위에서부터 **준비 셀**(제공 코드)을 먼저 실행하세요.
2. 각 문제는 **여러 단계**로 나뉩니다. 각 단계 셀의 지시를 따라 답안 셀을 채우고 자가채점으로 확인하세요.
3. 에이전트 라우팅은 답안 셀의 `tool_names(...)` **출력으로 관찰**하고(자가채점은 도구 값과 기록의 구조만 검사), 고친 도구 자체는 직접 불러 **값으로** 결정적으로 채점합니다.

화이팅!

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델. 18일차에서 배운 그대로입니다(이 셀은 실행만 하세요).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
# [제공 코드] 구청 민원 도구 재료. 표·공통 import (이 셀은 실행만 하세요)
import re
import pandas as pd
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.agents import create_agent
_fees = pd.read_csv('data/fees.csv')
FEE_TABLE = dict(zip(_fees['document'], _fees['fee']))
_dls = pd.read_csv('data/deadlines.csv')
DEADLINE_TABLE = dict(zip(_dls['minwon'], _dls['days']))

def tool_names(result):
    """메시지 기록에서 실제로 불린 도구 이름 목록을 뽑는다."""
    names = []
    for message in result['messages']:
        if isinstance(message, AIMessage):   # 도구 호출은 AIMessage 에만 담긴다
            for call in message.tool_calls:
                names.append(call['name'])
    return names
print('준비 완료')

---
# 1. 민원 안내 에이전트 (RAG + 계산 2종 + 출처)
**배경**: 구청 민원 창구를 돕는 에이전트를 만듭니다. **안내 검색·수수료 계산·처리기한 조회** 세 도구를 붙이고, 근거에 기반해 출처를 밝혀 답하게 합니다. 아래 **단계별로** 완성합니다.

In [ ]:
# [제공 코드] 검색기 준비. 15~16일차에 배운 벡터 검색을 함수 하나로 묶어 둡니다.
# (불러오는 데 잠시 걸립니다. 이 단원의 주제는 이 검색을 '도구'로 감싸 에이전트에 붙이는 것입니다.)
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer


def build_retriever(csv_path, text_columns, collection_name):
    """CSV 를 임베딩해 벡터DB에 넣고, 그 코퍼스를 검색하는 함수를 돌려준다(15~16일차 절차 그대로)."""
    embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')
    df = pd.read_csv(csv_path)
    texts = df[text_columns].agg(' '.join, axis=1).tolist()
    client = chromadb.EphemeralClient()
    # 같은 이름이 남아 있으면 지우고 새로 만든다(셀을 다시 실행해도 안전하게).
    if collection_name in [c.name for c in client.list_collections()]:
        client.delete_collection(collection_name)
    collection = client.create_collection(
        collection_name, metadata={'hnsw:space': 'cosine'})
    collection.add(ids=df['id'].tolist(),
                   embeddings=embedder.encode(texts, normalize_embeddings=True).tolist(),
                   documents=texts)

    def retrieve(query, k=2):
        """질문과 의미가 가장 가까운 문서 본문 k개를 리스트로 돌려준다."""
        hits = collection.query(
            query_embeddings=embedder.encode([query], normalize_embeddings=True).tolist(),
            n_results=k)
        return hits['documents'][0]

    return retrieve


retrieve_minwon = build_retriever('data/minwon_faq.csv', ['category', 'text'], 'minwon_faq_lv3')
print('검색기 준비 완료: retrieve_minwon')

### 1단계: 세 도구 만들기
다음 세 도구를 `@tool` 로 만드세요(문자열 입력은 `.strip()`).
- **`search_minwon(query)`**: `retrieve_minwon(query.strip(), 2)` 결과를 `'\n'` 으로 이어 반환(민원 안내 검색).
- **`calc_fee(document, count)`**: `FEE_TABLE` 단가 × 매수를 문자열로 반환(발급 수수료 계산).
- **`lookup_deadline(minwon)`**: `DEADLINE_TABLE` 의 일수를 `'N일'`(없으면 `'모름'`)로 반환(처리기한).

docstring 은 셋 다 **열 자 이상**으로, 괄호 안의 쓰임이 드러나게 적으세요 — `search_minwon` 에는 **`검색`** 또는 **`안내`**, `calc_fee` 에는 **`수수료`**, `lookup_deadline` 에는 **`기한`·`기간`·`며칠`** 중 하나가 들어가야 합니다(자가채점이 봅니다).

<details><summary>힌트</summary>

```text
접근방법:
- 세 함수 모두 @tool 을 붙이고 타입 힌트를 적는다. docstring 은 '언제 쓰는 도구인지' 가 서로
  겹치지 않게 쓴다 — 겹치면 에이전트가 어느 것을 골라야 할지 알 수 없다.

세부구현:
1. search_minwon: 제공된 retrieve_minwon 으로 상위 2개를 받아 줄바꿈으로 이어 붙인다.
2. calc_fee: FEE_TABLE 에서 서류 단가를 꺼내(없으면 0) 매수를 곱하고 문자열로 바꾼다.
3. lookup_deadline: DEADLINE_TABLE 에서 일수를 꺼낸다. 0 일이 들어와도 '모름' 으로 새지 않도록
   '값이 없는지' 를 None 과 비교해 판단하고, 있으면 뒤에 '일' 을 붙인다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
from langchain_core.tools import BaseTool
# 셋 다 @tool 로 만든 진짜 도구여야 에이전트에 붙일 수 있습니다(그냥 함수면 여기서 걸립니다).
assert all(isinstance(t, BaseTool) for t in [search_minwon, calc_fee, lookup_deadline])
# 검색 도구는 '무엇을 돌려주는지' 까지 봅니다 - 지어낸 문자열을 돌려주면 여기서 걸립니다.
assert search_minwon.invoke({'query': '전입신고'}) == '\n'.join(retrieve_minwon('전입신고', 2))
assert calc_fee.invoke({'document': '여권 복수', 'count': 1}) == '50000'
assert lookup_deadline.invoke({'minwon': '영업 신고'}) == '3일'
# docstring 이 서로 다른 쓰임을 말해 줘야 에이전트가 셋 중 하나를 고를 수 있습니다.
for _t in [search_minwon, calc_fee, lookup_deadline]:
    assert len(_t.description) >= 10, f'{_t.name} 의 docstring 을 열 자 이상 적으세요'
assert any(w in search_minwon.description for w in ['검색', '안내'])
assert '수수료' in calc_fee.description
assert any(w in lookup_deadline.description for w in ['기한', '기간', '며칠'])
print('✅ 통과!')

### 2단계: 에이전트 조립(출처 표기 프롬프트)
- 시스템 프롬프트 문자열을 변수 **`SYS_PROMPT`** 에 담으세요. **"도구로 확인한 내용에 근거해 답하라"** 는 지시와, 답 끝에 붙일 출처 문구 **`(안내: 구청 민원실)`** 이 **그 문자열 안에 그대로** 들어가야 합니다.
- 1단계의 **세 도구를 모두** 붙이고 `SYS_PROMPT` 를 준 에이전트를 만들어 **`minwon_agent`** 에 담으세요.

<details><summary>힌트</summary>

```text
세부구현:
1. SYS_PROMPT = '너는 구청 민원 도우미다. ... (안내: 구청 민원실) ...' 처럼 한 문자열로 적는다.
2. create_agent 에 model, 세 도구 리스트, system_prompt=SYS_PROMPT 를 넘긴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] - 출처 문구는 프롬프트 문자열에서 결정적으로 검사합니다(모델 답이 아니라).
from langchain_core.runnables import Runnable
assert isinstance(SYS_PROMPT, str)
assert '(안내: 구청 민원실)' in SYS_PROMPT   # 출처 문구를 프롬프트에 넣었는가
assert '근거' in SYS_PROMPT                  # 근거에 기반해 답하라고 지시했는가
# LangChain 이 만든 것은 모두 Runnable(invoke 로 부를 수 있는 실행 단위)입니다 -
# invoke 라는 이름만 흉내 낸 객체는 여기서 걸립니다.
assert isinstance(minwon_agent, Runnable)
print('✅ 통과! (도구가 정말 붙었는지는 3단계에서 확인합니다)')

### 3단계: 세 종류 질문을 넣고 라우팅 관찰하기
`minwon_agent` 에 아래 세 질문을 각각 넣어 결과를 `a_search`, `a_fee`, `a_dl` 에 담으세요. 질문마다 **알맞은 도구**가 불리는 것을 출력으로 볼 수 있습니다(자가채점은 세 응답 기록이 만들어졌는지만 봅니다).
- `a_search`: **'여권을 잃어버렸는데 어떻게 하나요?'** (→ 검색)
- `a_fee`: **'가족관계증명서 2장 수수료는?'** (→ 수수료)
- `a_dl`: **'자동차 등록은 며칠 걸리나요?'** (→ 처리기한)

<details><summary>힌트</summary>

```text
세부구현:
1. 각 질문을 minwon_agent.invoke({'messages':[HumanMessage(질문)]}) 로 실행한다.
2. 결과를 a_search, a_fee, a_dl 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] - 세 질문을 실제로 돌렸는지, 그리고 2단계 에이전트에 도구가 정말 붙었는지 봅니다.
# 질문마다 '어느' 도구가 불렸는지(라우팅)는 모델이 정하므로 위 출력으로 눈으로 확인하세요.
assert a_search['messages'] and a_fee['messages'] and a_dl['messages']
for question, res in [('여권을 잃어버렸는데 어떻게 하나요?', a_search),
                      ('가족관계증명서 2장 수수료는?', a_fee),
                      ('자동차 등록은 며칠 걸리나요?', a_dl)]:
    assert any(m.content == question for m in res['messages']), question
    # 모델이 만든 응답인지 - 손으로 지어낸 AIMessage 에는 응답 메타데이터가 없습니다.
    assert any(isinstance(m, AIMessage) and m.response_metadata
               for m in res['messages']), question
# 세 질문 전체에서 도구가 한 번도 안 불렸다면 2단계에서 도구를 안 붙인 것입니다.
all_calls = tool_names(a_search) + tool_names(a_fee) + tool_names(a_dl)
assert all_calls, '도구가 한 번도 불리지 않았습니다 - 2단계에서 세 도구를 붙였는지 확인하세요'
assert set(all_calls) <= {'search_minwon', 'calc_fee', 'lookup_deadline'}
print('✅ 통과! 불린 도구:', all_calls)

### 4단계: 구조화된 최종 응답을 받아 안내문으로 정리하기
**배경**: 실무에서는 에이전트의 답을 **줄글**이 아니라 **정해진 스키마**(민원 종류·안내문·출처)로 받아 화면·다음 시스템에 바로 넘깁니다. 지난 단원 과제에서 만나 본 **`response_format`**(pydantic `BaseModel`) 으로 최종 답을 구조화해 받는 것을, 아래 **제공 셀**이 시연합니다(`demo_answer` 는 `MinwonAnswer` 인스턴스). 여러분은 그 **구조화 응답을 소비**하는 함수를 만듭니다.

**요구사항**:
- 함수 **`format_notice(ans)`** 를 만드세요. `MinwonAnswer` 인스턴스 `ans` 를 받아 **`'[민원종류] 안내문 (출처: 출처)'`** 형태의 문자열을 돌려줍니다.

**예시**: `minwon_type='수수료 안내'`, `answer='2000원입니다.'`, `source='구청 민원실'` → `'[수수료 안내] 2000원입니다. (출처: 구청 민원실)'`

<details><summary>힌트</summary>

```text
세부구현:
1. ans.minwon_type · ans.answer · ans.source 를 f-문자열로 조립한다.
2. 형식: f'[{ans.minwon_type}] {ans.answer} (출처: {ans.source})'.
```

</details>

In [ ]:
# [제공 코드] 구조화된 출력 시연 - 지난 단원의 response_format 으로 최종 답을 스키마에 담습니다(실행만 하세요).
from pydantic import BaseModel, Field

@tool
def _demo_fee(document: str, count: int) -> str:
    """민원 서류 발급 수수료(원)를 계산한다."""
    return str(FEE_TABLE.get(document.strip(), 0) * count)

class MinwonAnswer(BaseModel):
    """민원 안내의 구조화된 최종 응답."""
    minwon_type: str = Field(description='민원 종류(수수료 안내·처리기한 안내·일반 안내 등)')
    answer: str = Field(description='사용자에게 줄 안내문 한 문장')
    source: str = Field(description='근거 출처')

_struct_agent = create_agent(model, [_demo_fee],
                             system_prompt='너는 구청 민원 도우미다. 도구로 확인해 답하고 출처를 밝혀라.',
                             response_format=MinwonAnswer)
demo_answer = _struct_agent.invoke(
    {'messages': [HumanMessage('가족관계증명서 2장 수수료는?')]})['structured_response']
print('구조화 응답 타입:', type(demo_answer).__name__)
print(' minwon_type =', demo_answer.minwon_type)
print(' answer      =', demo_answer.answer)
print(' source      =', demo_answer.source)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] - 구조화 응답을 소비하는 함수를 결정적으로 검사합니다(모델 없이, 직접 만든 인스턴스로).
sample_answer = MinwonAnswer(minwon_type='수수료 안내', answer='2000원입니다.', source='구청 민원실')
assert format_notice(sample_answer) == '[수수료 안내] 2000원입니다. (출처: 구청 민원실)'
assert set(MinwonAnswer.model_fields) == {'minwon_type', 'answer', 'source'}   # 스키마 필드 확인
print('✅ 통과!')

---
# 2. 고장난 도구 세 개를 진단하고 고치기
**배경**: 구청 안내 창구에 도구 세 개짜리 에이전트를 붙였는데 **엉뚱하게 굴러갑니다.** 제공 셀에 **고장난 도구 세 개**와 그것으로 실제로 나온 **메시지 기록**이 있습니다. 기록을 읽어 무엇이 잘못됐는지 짚고, 도구를 고쳐 세 종류 질문이 **제자리로 가게** 만드세요.

이 문제는 **모델이 만드는 답을 채점하지 않습니다.** 채점하는 것은 두 가지입니다 — **고친 도구가 혼자서도 옳게 동작하는가**(결정적), 그리고 **에이전트가 옳은 도구를 불렀는가**(구조).

In [ ]:
# [제공 코드] 고장난 도구 세 개 - 이 셀은 실행만 하세요.
from langchain_core.tools import tool

_OFFICE_HOURS = {'본청': '09:00-18:00', '동주민센터': '09:00-17:00'}


@tool
def handle(query: str) -> str:
    """처리한다."""
    return '처리 완료'


@tool
def office_hours(place: str) -> str:
    """청사 운영 시간을 돌려준다."""
    return _OFFICE_HOURS.get(place, '없음')          # 받은 글자를 그대로 열쇠로 쓴다


@tool
def check_status(number: str) -> str:
    """민원 접수 번호로 처리 상태를 확인한다. 정확한 결과를 위해 여러 번 확인하는 것이 좋다."""
    return f'{number}: 처리중'


print('고장난 도구:', [t.name for t in [handle, office_hours, check_status]])

In [ ]:
# [제공 코드] 이 도구들로 실제로 나온 기록 - 고정해 두었습니다(실행마다 달라지지 않게).
#  각 항목은 (질문, 불린 도구 목록, 도구에 넘어간 인자) 입니다.
BROKEN_TRACES = [
    ('주민등록등본 수수료가 얼마인가요?', ['handle'], [{'query': '주민등록등본 수수료'}]),
    ('동주민센터 몇 시까지 하나요?', ['office_hours'], [{'place': '동주민센터는'}]),
    ('접수번호 A-1024 어떻게 됐나요?', ['check_status'] * 3,
     [{'number': 'A-1024'}, {'number': 'A-1024'}, {'number': 'A-1024'}]),
]

for question, called, args in BROKEN_TRACES:
    print(f'질문: {question}')
    print(f'  불린 도구  : {called}')
    print(f'  넘어간 인자: {args}')
    print()

### 1단계: 진단하기 (서술형)

세 기록을 하나씩 읽고, 각각 **어떤 증상**이고 **무엇이 원인**인지 아래 markdown 셀에 적으세요. 교안 5절의 증상 표를 옆에 두고 보면 됩니다.

각 기록마다 두 가지를 적습니다.

1. 증상 이름 (안 부름 / 잘못 고름 / 값은 맞는데 결과가 없음 / 과다 호출 중 하나)
2. 그 도구의 **어느 부분**이 원인인지 (이름·docstring·인자 처리·반환값 중에서)

**답안** *(아래에 세 기록의 진단을 서술하세요)*

*(여기에 자신의 진단을 서술하세요)*

### 2단계: 도구 고치기

세 도구를 고쳐 새로 만드세요. 이름은 아래로 **정확히** 맞춰야 채점됩니다.

- **`fee_guide(document: str, count: int) -> str`** — `handle` 을 대체합니다. `FEE_TABLE` 에서 서류 단가를 찾아 매수를 곱하고 **`'{서류} {매수}장 = {금액}원'`** 형식으로 돌려주세요. docstring 에 *언제 쓰는 도구인지*(**`수수료`** 라는 말이 들어가게)와 **허용 서류 이름**(적어도 **`주민등록등본`**)을 적으세요. 없는 서류면 안내 문자열을 돌려줍니다.
- **`office_hours_fixed(place: str) -> str`** — 인자를 **다듬으세요**. 조사(`'는'`·`'은'`)와 앞뒤 공백을 떼어 낸 뒤 `_OFFICE_HOURS` 에서 찾습니다. 없으면 **안내 문자열**을 돌려줍니다(`'없음'` 금지). docstring 에 허용 값(**`본청`·`동주민센터`**)을 적으세요.
- **`check_status_fixed(number: str) -> str`** — 반복을 부추기는 문구(`'여러 번'`)를 빼고, 결과가 **`확정`** 임을 docstring 에 그 말로 밝히세요. 반환은 `'{번호}: 처리중 (확인 완료)'` 형식입니다.

세 docstring 은 모두 **스무 자 이상**이어야 합니다 — 자가채점이 위에 굵게 적은 낱말들과 길이를 함께 봅니다(설명이 곧 라우팅의 근거이므로, 지워 버리는 것은 고치는 것이 아닙니다).

**예시**: `office_hours_fixed('동주민센터는')` 은 `'09:00-17:00'` 을 포함한 문자열을 돌려줍니다. `fee_guide('주민등록등본', 3)` 은 `'주민등록등본 3장 = 1200원'` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 세 가지를 각각 다르게 고친다. 하나는 이름·설명, 하나는 인자 다듬기, 하나는 설명 속 지시문이다.

세부구현:
1. fee_guide
   1-1. 서류 이름의 앞뒤 공백을 떼고 표에서 단가를 찾는다.
   1-2. 없으면 고를 수 있는 서류를 알려 주는 문장을 돌려준다.
   1-3. 있으면 단가에 매수를 곱해 지정된 형식으로 만든다.
2. office_hours_fixed
   2-1. 앞뒤 공백을 떼고, 끝에 붙은 조사 한 글자를 떼어 낸다.
   2-2. 표에 없으면 고를 수 있는 곳을 알려 주는 문장을 돌려준다.
3. check_status_fixed
   3-1. docstring 에서 반복을 권하는 문장을 빼고 결과가 확정임을 적는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 도구 자체는 모델을 거치지 않고 직접 부를 수 있다 - 값을 정확히 검사한다.
assert fee_guide.invoke({'document': '주민등록등본', 'count': 3}) == '주민등록등본 3장 = 1200원'
assert '없' in fee_guide.invoke({'document': '없는서류', 'count': 1})
# 인자 다듬기 - 조사가 붙어도, 공백이 끼어도 같은 답이 나와야 한다.
for surface in ['동주민센터', '동주민센터는', ' 동주민센터 ']:
    got = office_hours_fixed.invoke({'place': surface})
    assert '09:00-17:00' in got, f'{surface!r} 를 못 다듬었습니다'
# 없는 값에 '없음' 만 돌려주면 조용한 실패가 된다 - 무엇을 고르라는 안내가 있어야 한다.
_miss = office_hours_fixed.invoke({'place': '구청앞카페'})
assert _miss != '없음' and ('본청' in _miss or '동주민센터' in _miss)
assert check_status_fixed.invoke({'number': 'A-1024'}) == 'A-1024: 처리중 (확인 완료)'
# 설명은 모델에게 주는 지시문이다 - 지워 버리면 라우팅 근거까지 사라진다.
# 그래서 '반복 문구가 빠졌는가'(부정)와 '무엇을 적었는가'(긍정)를 함께 본다.
for _t, _words in [(fee_guide, ['수수료', '주민등록등본']),
                   (office_hours_fixed, ['본청', '동주민센터']),
                   (check_status_fixed, ['확정'])]:
    assert len(_t.description) >= 20, f'{_t.name} 의 docstring 을 스무 자 이상 적으세요'
    for _w in _words:
        assert _w in _t.description, f"{_t.name} 의 docstring 에 '{_w}' 가 들어가야 합니다"
assert '여러 번' not in check_status_fixed.description
print('✅ 통과!')

### 3단계: 고친 도구로 라우팅 확인

고친 세 도구로 에이전트를 만들어 세 질문이 제자리로 가는지 확인하세요.

- `create_agent(model, [fee_guide, office_hours_fixed, check_status_fixed])` 로 **`fixed_agent`** 를 만드세요.
- `BROKEN_TRACES` 의 **질문 세 개**를 차례로 `invoke` 하고, 각 결과의 **불린 도구 이름 목록**을 리스트 **`fixed_calls`** 에 순서대로 담으세요(각 항목이 그 질문에서 불린 도구 이름들의 리스트).
- 질문마다 불린 도구와 최종 답을 출력하세요.

**예시**: `fixed_calls[0]` 에는 `'fee_guide'` 가 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 기록에서 도구 이름을 뽑는 헬퍼는 맨 위 제공 셀에 이미 있다.

세부구현:
1. 고친 도구 세 개로 에이전트를 만든다.
2. 고정해 둔 기록 목록에서 질문만 꺼내 차례로 넣는다.
3. 결과마다 도구 이름 목록을 뽑아 모으고, 함께 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 모델이 만드는 답 문장은 실행마다 달라지므로 '어떤 도구가 불렸나' 라는 구조만 본다.
assert len(fixed_calls) == 3, '세 질문의 결과를 순서대로 담으세요'
assert 'fee_guide' in fixed_calls[0], '수수료 질문은 fee_guide 로 가야 합니다'
assert 'office_hours_fixed' in fixed_calls[1], '운영 시간 질문은 office_hours_fixed 로 가야 합니다'
assert 'check_status_fixed' in fixed_calls[2], '접수 상태 질문은 check_status_fixed 로 가야 합니다'
# 과다 호출이 줄었는지 - 원래 세 번 불리던 것이 이제 한 번(많아도 두 번)이면 된다.
# 정확히 몇 번 부를지는 모델이 정하므로 여유를 두고, '반복을 권하는 문구를 뺐는가' 는
# 2단계에서 docstring 으로 이미 결정적으로 검사했다.
assert fixed_calls[2].count('check_status_fixed') <= 2, \
    '상태 확인이 아직 여러 번 불립니다 - docstring 에 반복을 권하는 말이 남았는지 보세요'
print('✅ 통과! 불린 도구:', fixed_calls)

---
수고했어요! LV3 에서 이 단원의 개념을 **민원 안내 에이전트**(멀티툴 라우팅+출처)와 **라우팅 진단·수리**(기록을 읽고 도구를 고치기)로 통합했습니다. 도구를 **쓰는** 데서 나아가 잘못 굴러가는 에이전트를 **진단하고 고치는** 데까지 왔습니다. 다음 단원에서는 에이전트로 **데이터 분석을 자동화**합니다.